# ISOM 839 · Session 9 — Where Do Customers Go?

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Every customer is somewhere on a journey: active, wobbling, gone quiet, gone for good. Tonight you put that journey into **one small table**, use it to see the future, and turn it into **dollars**.

### 🗺️ Tonight's road map: see it on a slide, then do it here

| Part | What you do | Time |
|---|---|---|
| **0 · Setup** | Run one cell | 1 min |
| **1 · Where do customers go?** | Build the table. Move 10,000 customers forward a year. | 15 min |
| **2 · The long run** | Watch a chain forget where it started | 5 min |
| **3 · Customers in dollars** | Live one customer's life, then 10,000. Lifetime value. Pick the better of two campaigns. | 25 min |
| **Bonus · PageRank** | Google's original idea in 8 lines | 5 min |
| **Homework #6** | The same steps on 5,000 streaming subscribers | at home |

### How to use this notebook
1. Run the cells **top to bottom** with **Shift + Enter**.
2. After every code cell there is a **✅ What you should see** box. If your output matches, move on.
3. You never need to type anything until a cell says **🔧 Your turn**.

---
## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 2)
print('Ready!')

**✅ What you should see:** `Ready!`

---
## Part 1 — Where do customers go?

**The business:** an app that charges **\$15 a month**. Each month, every customer is in one of four **states**:

| State | What it means | Pays this month? |
|---|---|---|
| **Active** | used the app in the last 30 days | \$15 |
| **At-Risk** | still paying, but using it half as much | \$15 |
| **Lapsed** | paused, not paying | \$0 |
| **Churned** | cancelled for good | \$0, forever |

**The goldfish rule (the Markov property):** where a customer goes next month depends **only on where they are now**, not on their whole history. Like a board game: your next square depends on your square and the dice, not on how you got there.

### Step 1.1 — Write the table

One row per state. Each row says **where people in that state go next month**. Every row must add up to 1, because everyone has to go *somewhere*.

In [ ]:
states = ['Active', 'At-Risk', 'Lapsed', 'Churned']

P = np.array([
    # Active  At-Risk  Lapsed  Churned     <- where they go NEXT month
    [ 0.80,   0.15,    0.03,   0.02 ],   # from Active
    [ 0.30,   0.40,    0.20,   0.10 ],   # from At-Risk
    [ 0.10,   0.10,    0.50,   0.30 ],   # from Lapsed
    [ 0.00,   0.00,    0.00,   1.00 ],   # from Churned (nobody comes back)
])

print(pd.DataFrame(P, index=states, columns=states))
print('\nRow sums:', P.sum(axis=1))
assert np.allclose(P.sum(axis=1), 1), 'A row does not add up to 1. Check your numbers!'

**✅ What you should see:** the 4 × 4 table, then `Row sums: [1. 1. 1. 1.]`.

Read the first row out loud: *"Of every 100 active customers, 80 stay active, 15 start to wobble, 3 go quiet, and 2 leave."* The `assert` line is a safety check: a row that does not add up to 1 is the most common bug in real Markov work, so we test for it first.

### Step 1.2 — Move everyone forward one month

`x` is a list of how many customers are in each state. **`x @ P`** means *"every customer takes one step using the table."*

By hand, for Active next month: 10,000 × 0.80 (stayed) + 0 × 0.30 + 0 × 0.10 = **8,000**. Python does that for all four states at once.

In [ ]:
x = np.array([10_000, 0, 0, 0])      # month 0: 10,000 new customers, all Active
x1 = x @ P                            # everyone takes one step
print('Month 1:', dict(zip(states, x1.astype(int).tolist())))

**✅ What you should see:** `Active 8000 · At-Risk 1500 · Lapsed 300 · Churned 200`. Together they still add to 10,000. Nobody disappears; they just move.

### Step 1.3 — Do it 36 times (and get the answer to the class vote)

A loop repeats the step: month 1, month 2, … month 36.

In [ ]:
x = np.array([10_000, 0, 0, 0])
history = [x]
for month in range(36):
    x = x @ P                         # one more month
    history.append(x)

history = pd.DataFrame(history, columns=states).round().astype(int)
print(history.loc[[0, 1, 3, 6, 12, 24, 36]])

history.plot.area(figsize=(8, 3.8), color=['tab:green', 'gold', 'tab:orange', 'tab:red'], alpha=.85)
plt.xlabel('month'); plt.ylabel('customers'); plt.title('Where 10,000 customers go')
plt.show()

**✅ What you should see:** at **month 12, 2,829 are still Active**. That is answer **C** from the class vote. On the chart, the red Churned area slowly eats everything.

80% a month *sounds* great. Compounded, it is a **leaky bucket**. And notice: if nobody ever came back, only 0.8¹² × 10,000 = **687** would be active. The way back from At-Risk and Lapsed is worth about **2,100 customers**.

### 🔧 Your turn (2 minutes)
What if Active customers were a little happier? The cell below changes the Active row to **stay 0.85, wobble 0.10**. Run it. Then try your own numbers (keep the row adding to 1).

In [ ]:
P_test = P.copy()
P_test[0] = [0.85, 0.10, 0.03, 0.02]     # <- the new Active row: try your own numbers

x = np.array([10_000, 0, 0, 0])
for month in range(12):
    x = x @ P_test
print('Active after 12 months:', round(x[0]), '   (it was 2,829)')

**✅ What you should see:** with 0.85 / 0.10, about **3,555** active after a year. Five points of monthly retention keep 700+ more customers.

---
## Part 2 — The long run

### Step 2.1 — A chain forgets where it started

A made-up Boston coffee market. Each month, **90%** of Dunkin' regulars stay (10% switch to Starbucks). Starbucks regulars: **80%** stay (20% switch to Dunkin'). We start from three very different places and run each for 24 months.

In [ ]:
coffee = np.array([[0.90, 0.10],      # from Dunkin':   stay, switch
                   [0.20, 0.80]])     # from Starbucks: switch, stay

plt.figure(figsize=(8, 3.6))
for start in ([1.0, 0.0], [0.5, 0.5], [0.0, 1.0]):
    share = np.array(start)
    path = [share[0]]
    for month in range(24):
        share = share @ coffee
        path.append(share[0])
    plt.plot(path, lw=3, label=f"start: {start[0]:.0%} Dunkin'")

plt.axhline(2/3, ls='--', c='gray'); plt.ylim(0, 1)
plt.xlabel('month'); plt.ylabel("Dunkin' share"); plt.title('Three starts, one ending'); plt.legend()
plt.show()
print(f"Dunkin' share after 24 months: {share[0]:.1%}")

**✅ What you should see:** three lines that start far apart and all land on the dashed line at **66.7%**. That ending is called the **steady state**. Here it is 0.20 ÷ (0.10 + 0.20) = ⅔.

**The business lesson:** long-run market share is set by **switching rates**, not by today's share.

### Step 2.2 — With a one-way door, the long run is boring

Our app has Churned, which nobody leaves. So in the long run, *everyone* ends up there. The more useful question is **how long that takes**.

In [ ]:
x = np.array([10_000, 0, 0, 0])
month = 0
while x[3] < 9_999:                   # keep going until 99.99% have churned
    x = x @ P
    month += 1
print('Months until 99.99% have churned:', month)

half = (history['Churned'] >= 5_000).idxmax()
print('Month when half the customers are gone:', half)

**✅ What you should see:** about **116 months** until (almost) everyone has churned, and **half are gone by month 11**.

"Where do they end up?" has a boring answer (Churned). The money question is: **how many months do they pay before they leave?** That is Part 3.

---
## Part 3 — Customers in dollars

### Step 3.0 — Meet Maya: live one customer's life

Maya just signed up, so she starts **Active**. Each month we roll the dice using her row of the table, exactly like the dots on the slide, until she churns. Every month she is Active or At-Risk, she pays \$15.

Her life is printed as letters: **A** = Active, **R** = At-Risk, **L** = Lapsed.

In [ ]:
rng = np.random.default_rng(839)              # fixed dice, so your Maya matches mine

def live_one_life():
    state, paid, life = 0, 0, ''              # start Active, $0 paid so far
    while state != 3:                         # until she churns
        life += 'ARL'[state]                  # write down this month
        paid += [15, 15, 0][state]            # Active and At-Risk pay $15, Lapsed pays $0
        state = rng.choice(4, p=P[state])     # roll the dice with her row of the table
    return paid, life

paid, life = live_one_life()
print("Maya's life:", life)
print(f'Months: {len(life)}    She paid: ${paid}')

**✅ What you should see:** `Maya's life: AAAAAAA`, 7 months, **\$105**. Seven happy months, then she cancelled.

Is Maya typical? There is no way to know from one life. So let's live **10,000** of them.

In [ ]:
lives = np.array([live_one_life()[0] for _ in range(10_000)])    # takes a couple of seconds

print(f'Average customer:  ${lives.mean():,.2f}')
print(f'Median customer:   ${np.median(lives):,.0f}')
print(f'Paid $60 or less:  {(lives <= 60).mean():.0%}')

plt.figure(figsize=(8, 3.4))
plt.hist(lives, bins=range(0, 1000, 30), color='tab:orange', edgecolor='white')
plt.axvline(lives.mean(), color='black', ls='--', label=f'average ${lives.mean():.0f}')
plt.xlabel('what one customer paid over her whole life ($)'); plt.ylabel('customers'); plt.legend()
plt.show()

**✅ What you should see:** an average of about **\$192**, a median of about **\$135**, and roughly **24%** of customers paying \$60 or less. The histogram is lopsided: lots of short lives, a long tail of loyal customers.

**Almost nobody is worth exactly the average.** But the average is what a business plans with. Next: a shortcut that gives the average **without rolling a single die**.

### Step 3.1 — The shortcut: how many months does a customer stay, on average?

**Warm-up with one state.** If you stay with probability 0.8 each month, the expected stay is 1 ÷ (1 − 0.8) = **5 months**.

**With several states** that pass customers back and forth, there is a matrix version of the same formula:

* **Q** = the table without the Churned row and column (only the states where customers still count)
* **I** = the identity matrix, the matrix version of the number 1
* **N = (I − Q)⁻¹**, the matrix version of 1 ÷ (1 − r)

Row 0 of N tells you how many months a new Active customer spends in each state before churning.

In [ ]:
print('Warm-up: 1 / (1 - 0.8) =', round(1 / (1 - 0.8), 2), 'months\n')

Q = P[:3, :3]                          # keep Active, At-Risk, Lapsed; drop Churned
N = np.linalg.inv(np.eye(3) - Q)       # the matrix version of 1 / (1 - r)
months = N[0]                          # row 0 = a customer who starts Active

print('Expected months in each state:')
print(pd.Series(months, index=states[:3]).round(2))
print('\nTotal expected lifetime:', round(months.sum(), 1), 'months')

**✅ What you should see:** Active **10.07**, At-Risk **2.81**, Lapsed **1.73** months. Total lifetime **14.6 months**.

### Step 3.2 — Turn months into dollars

Multiply the months in each state by what that state pays.

In [ ]:
pay = np.array([15, 15, 0])            # $ per month in Active, At-Risk, Lapsed
clv = months @ pay                     # (10.07 x 15) + (2.81 x 15) + (1.73 x 0)
print(f'Lifetime value of a new customer: ${clv:,.2f}')

**✅ What you should see:** **\$193.17**. The 10,000 lives averaged about \$192: same answer, no dice. That is the most you should ever pay to win one new customer.

### Step 3.3 — The \$2 decision

Two campaigns. Each costs **\$2 per targeted customer per month**.

| Campaign | What it changes | Who you pay for |
|---|---|---|
| **A · Keep the Active happy** | Active stays 0.80 → **0.85** | every month a customer is **Active** |
| **B · Rescue the At-Risk** | At-Risk recovers 0.30 → **0.40** | only months a customer is **At-Risk** |

The function below does three things for a campaign: the **gain** in lifetime value, the **cost** (\$2 × months in the state you pay for), and the **net** (gain − cost). You don't need to change it.

In [ ]:
def campaign(P_new, target, cost_per_month=2.0):
    # target: 0 = you pay for Active months, 1 = you pay for At-Risk months
    months_new = np.linalg.inv(np.eye(3) - P_new[:3, :3])[0]
    gain = months_new @ pay - clv
    cost = cost_per_month * months_new[target]
    return round(gain, 2), round(cost, 2), round(gain - cost, 2)

P_A = P.copy();  P_A[0] = [0.85, 0.10, 0.03, 0.02]    # A: Active stays 0.80 -> 0.85
P_B = P.copy();  P_B[1] = [0.40, 0.30, 0.20, 0.10]    # B: At-Risk recovers 0.30 -> 0.40

results = pd.DataFrame([campaign(P_A, target=0), campaign(P_B, target=1)],
                       index=['A: keep Active happy', 'B: rescue At-Risk'],
                       columns=['CLV gain $', 'cost $', 'net $'])
print(results)

**✅ What you should see:**

| | CLV gain | cost | net |
|---|---|---|---|
| A | 30.83 | 25.11 | **5.71** |
| B | 12.90 | 5.25 | **7.64** |

**The plot twist.** On the gain alone, **A wins** (it's the bigger lever). After the cost, **B wins**: A pays \$2 on about 12.6 Active months per customer, B only on about 2.6 At-Risk months. **Spend where customers wobble, not on everyone.**

### 🔧 Your turn (2 minutes)
At what price does the winner switch? Run the cell, then add your own prices to the list.

In [ ]:
for cost in [0, 1.00, 1.50, 2.00, 3.00]:          # <- add your own prices
    net_A = campaign(P_A, 0, cost)[2]
    net_B = campaign(P_B, 1, cost)[2]
    winner = 'A' if net_A > net_B else 'B'
    print(f'${cost:.2f} a month:   net A {net_A:7.2f}   net B {net_B:7.2f}   ->  {winner} wins')

**✅ What you should see:** A wins at \$0, \$1.00 and \$1.50. B wins at \$2.00 and \$3.00. The switch happens at about **\$1.81** a month.

---
## Bonus — PageRank: Google's original idea in 8 lines

Same math, different states. The states are **web pages**. A random surfer clicks a random link on the page 85% of the time, and jumps to any page 15% of the time. The share of time the surfer spends on each page, in the long run, **is** the ranking.

In [ ]:
pages = ['Blog', 'Review', 'Shop', 'News', 'Hobby']
links = {'Blog': ['Review', 'Shop'], 'Review': ['Shop'], 'Shop': ['Review', 'Blog'],
         'News': ['Blog', 'Review', 'Shop'], 'Hobby': ['News']}

W = np.zeros((5, 5))
for i, page in enumerate(pages):
    for target in links[page]:
        W[i, pages.index(target)] = 1 / len(links[page])   # click one of the page's links at random

G = 0.85 * W + 0.15 / 5                # 15% of the time: jump to any page
x = np.ones(5) / 5                     # the surfer starts anywhere
for step in range(100):
    x = x @ G                          # the same one line as Part 1
print((pd.Series(x, index=pages).sort_values(ascending=False) * 100).round(1).astype(str) + '%')

**✅ What you should see:** Shop **39.6%**, Review **30.5%**, Blog **21.4%**, News **5.6%**, Hobby **3.0%**.

Review and Shop both have 3 pages linking to them, yet Shop wins. Why? Review sends **all** of its votes to Shop. It is not how many pages link to you. It is **who**.

---
# Homework #6 — Churn Markov Model (Streaming Subscribers) 📺

**Submit on Canvas:** this notebook completed, plus your go / no-go paragraph in the text box. Due date and rubric are on the Canvas assignment.

**The data.** A streaming service followed **5,000 new subscribers** for 12 months. Each month, every subscriber is in one of five states:

| State | Meaning |
|---|---|
| **New** | first month only |
| **Engaged** | watching a lot |
| **Casual** | watching a little |
| **Dormant** | still paying, not watching |
| **Cancelled** | gone for good (like Churned tonight) |

Every state except Cancelled pays a **\$12.99 monthly margin**. The file has one row per subscriber and one column per month.

You already did every step tonight. Each step below says **which part of tonight to copy from**.

In [ ]:
cohort = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-839/master/public/session-09/streaming_cohort.csv')
hw_states = ['New', 'Engaged', 'Casual', 'Dormant', 'Cancelled']
month_cols = [c for c in cohort.columns if c.startswith('month_')]

print(cohort.head())
print('\nsubscribers:', len(cohort), '· months observed:', len(month_cols) - 1)
print('\nstate mix at month 12:')
print(cohort['month_12'].value_counts().reindex(hw_states).fillna(0).astype(int))

**✅ What you should see:** the first five subscribers, `subscribers: 5000 · months observed: 12`, and how many are in each state at month 12.

### HW Step 1 — Estimate the table from the data

Every subscriber, every month, makes one move: from this month's state to next month's. The cell below lines up **all** those moves for you in two columns, `from` and `to`. Your job is to **count** them and turn the counts into a table.

In [ ]:
# Done for you: one row per month-to-month move (5,000 subscribers x 12 moves = 60,000 rows)
moves = pd.concat([pd.DataFrame({'from': cohort[month_cols[t]], 'to': cohort[month_cols[t + 1]]})
                   for t in range(len(month_cols) - 1)], ignore_index=True)
print(moves.head())
print('moves:', len(moves))

**Hints for the TODO below:**
1. `pd.crosstab(moves['from'], moves['to'])` counts how many moves go from each state to each state.
2. Add `.reindex(index=hw_states, columns=hw_states, fill_value=0)` so the rows and columns are in the right order (nobody ever moves *into* New, so that column would otherwise be missing).
3. Divide each row by its total: `counts.div(counts.sum(axis=1), axis=0)`.
4. Check the Cancelled row is `0 0 0 0 1`. Nobody comes back.

In [ ]:
# TODO 1: count the moves, divide each row by its total, and call the result P_hw (a 5 x 5 numpy array)
counts = ...
P_hw = ...

print(pd.DataFrame(P_hw, index=hw_states, columns=hw_states).round(3))
assert np.allclose(P_hw.sum(axis=1), 1), 'A row does not add up to 1'

### HW Step 2 — Project 24 months forward (copy from Step 1.3)

Start with 5,000 New subscribers, `x = np.array([5_000, 0, 0, 0, 0])`, take 24 steps with `x @ P_hw`, and plot how many are in each state over time.

In [ ]:
# TODO 2: project 5,000 New subscribers 24 months forward and plot the five states over time

### HW Step 3 — Lifetime value per new subscriber (copy from Steps 3.1 and 3.2)

This time **four** states still count (New, Engaged, Casual, Dormant), so `Q = P_hw[:4, :4]` and `np.eye(4)`. All four pay \$12.99, so lifetime value = 12.99 × total expected months. Show **N's first row** and the dollar figure, not just the number.

In [ ]:
# TODO 3: N = (I - Q)^-1 over the four paying states; LTV = 12.99 x expected months, starting New

### HW Step 4 — The win-back campaign: go or no-go? (copy from Step 3.3)

Management wants to email Dormant subscribers. They expect the **Dormant → Engaged** probability to rise from about 0.05 to about **0.15** (take the extra 0.10 out of Dormant → Dormant, so the row still adds to 1). It costs **\$3 per Dormant subscriber per month**.

1. Copy `P_hw`, change the Dormant row, and recompute lifetime value.
2. Gain = new LTV − old LTV.
3. Cost = \$3 × expected months in Dormant (from the *new* N).
4. Net = gain − cost. Then write your **go / no-go** with the numbers, and name **the one assumption that could flip your answer**.

In [ ]:
# TODO 4: modify the Dormant row, recompute LTV, compute gain, cost, and net; then decide go / no-go

---
### Submission checklist
- [ ] Estimated table printed; rows add to 1; Cancelled row is `0 0 0 0 1`
- [ ] 24-month projection plotted
- [ ] Lifetime value shown as a calculation (N first, then dollars)
- [ ] Go / no-go paragraph with the gain, the cost, and the assumption that could flip it, pasted into the Canvas text box

**Next class: Wednesday, November 18** (no class November 11, Veterans Day). We open with the **midterm case briefings**, then chains learn to choose: states, actions, rewards, and the Bellman equation. 🎯